# 05 · LSTM：单词级下一个 token 预测

本 Notebook 可独立运行，复用前三个 Notebook 对应的公共模块。
默认配置与另外两个模型一致，区别仅在循环层。
目标不是判断整段文本属于哪个类别，而是在每一个时间步预测下一个单词。


In [ ]:
import os
import sys
from pathlib import Path

# 支持从项目根目录或 notebooks 目录启动。
# Kaggle 上可在这里显式设置：
# os.environ["TIME_MACHINE_PROJECT_ROOT"] = "/kaggle/working/Time_Machine"
hint = os.environ.get("TIME_MACHINE_PROJECT_ROOT")
if hint:
    candidates = [Path(hint).expanduser().resolve()]
else:
    candidates = []
    for parent in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([parent / "Time_Machine", parent])
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "config.py").is_file() and (path / "src/generation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请在 Time_Machine 项目内启动，或设置 TIME_MACHINE_PROJECT_ROOT。")
existing_config = sys.modules.get("config")
if existing_config and Path(existing_config.__file__).resolve() != PROJECT_ROOT / "config.py":
    raise RuntimeError("kernel 中已经导入其他项目的 config，请重启 kernel 再运行本 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from config import ExperimentConfig, get_device, resolve_data_path, resolve_output_dir

data_path = resolve_data_path()
output_dir = resolve_output_dir()
device = get_device("auto")
print("项目根目录:", PROJECT_ROOT)
print("数据文件:", data_path)
print("输出目录:", output_dir)
print("计算设备:", device)


## 1. 共享参数、固定随机种子、准备数据

修改实验设置时，三个模型使用相同设置。默认训练 20 epochs。
初次检查可以将 epochs 改为 1，并使用单独输出目录，避免覆盖正式结果。


In [ ]:
from src.dataset import prepare_data
from src.engine import seed_everything, fit

config = ExperimentConfig()
seed_everything(config.seed)
data = prepare_data(data_path, config)
vocab = data.vocab
display(pd.DataFrame(data.metadata["splits"]).T)
print("词表大小:", len(vocab))
print("实验参数:", config)


## 2. 模型结构

LSTM 在每个时间步维护两个状态：隐藏状态 $h_t$ 和记忆单元 $c_t$。
遗忘门控制旧记忆保留多少，输入门控制新信息写入多少，输出门控制向外输出多少。

$$c_t = f_t \odot c_{t-1} + i_t \odot g_t$$
$$h_t = o_t \odot \tanh(c_t)$$

`nn.LSTM` 返回 `output, (hidden, cell)`。下一词 logits 仍使用每个时间步的 output；
Embedding、输出层、数据、损失和训练代码与其他模型共用。


In [ ]:
import torch
from torch import nn
from src.model_LSTM import TimeMachineLSTM

model = TimeMachineLSTM(
    vocab_size=len(vocab),
    embed_size=config.embed_size,
    hidden_size=config.hidden_size,
    num_layers=config.num_layers,
    dropout=config.dropout,
).to(device)
print(model)
print("参数量:", sum(parameter.numel() for parameter in model.parameters()))


## 3. 前向传播：查看每一步的形状

不使用双向循环层，以免未来单词泄漏给当前预测。
隐藏状态用于理解循环层和后续生成；输出层对 output 的所有时间步逐一作用。


In [ ]:
X, y = next(iter(data.loaders["valid"]))
X, y = X.to(device), y.to(device)
model.eval()
with torch.no_grad():
    embedded = model.embedding(X)
    recurrent_output, state = model.recurrent(embedded)
    logits, _ = model(X)
print("输入:", X.shape)
print("Embedding:", embedded.shape)
print("循环层 output:", recurrent_output.shape)
print("词表 logits:", logits.shape)
hidden, cell = state
print("最终 hidden:", hidden.shape)
print("最终 cell:", cell.shape)  # 二者均为 [num_layers, batch, hidden_size]


## 4. 多分类交叉熵

每个时间步是在整个词表中选择一个单词，目标为 `torch.long` 的单词 ID。
`[B,T,V]` 展开成 `[B*T,V]`，`[B,T]` 展开成 `[B*T]`。
直接传入原始 logits，不先 sigmoid 或 softmax。


In [ ]:
criterion = nn.CrossEntropyLoss()
loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
print("展平后的 logits:", logits.reshape(-1, logits.size(-1)).shape)
print("展平后的 target:", y.reshape(-1).shape)
print("尚未训练的 loss:", loss.item())
print("尚未训练的 perplexity:", torch.exp(loss).item())


## 5. 训练、验证并保存最佳模型

`fit()` 调用共享的 `train_one_epoch()` 和 `evaluate()`：Adam、梯度裁剪、
按目标 token 数量加权的平均损失。每个 batch 重置隐藏状态。
按验证集 Loss 保存最佳 epoch；训练结束后重新加载最佳权重，再评估一次测试集。
同一输出目录内重跑同一模型会覆盖该模型的上一轮结果。


In [ ]:
result = fit(model, data, config, device, output_dir)
history = pd.DataFrame(result["history"])
display(history)
print("最佳模型:", result["checkpoint_path"])


## 6. 学习曲线与最终指标

困惑度是 `exp(token 平均交叉熵)`，越低越好。
`accuracy` 包含 `<unk>`；`known_accuracy` 排除目标为 `<unk>` 的位置。
结合 `unknown_target_rate` 阅读指标，避免把预测未知词当作真实单词预测效果。
测试结果用于最终报告，调参主要看验证集。


In [ ]:
from src.plotting import plot_history

figure = plot_history(result["history"])
figure.savefig(result["run_dir"] / "learning_curves.png", dpi=150)
plt.show()
display(pd.DataFrame({
    "best_valid": result["metrics"]["valid"],
    "test": result["metrics"]["test"],
}).T)
print("最佳 epoch:", result["metrics"]["best_epoch"])


## 7. 从 checkpoint 重新加载并预测下一词

checkpoint 同时保存权重、词表、模型和实验配置，因此不需要重新构建词表。
候选展示屏蔽 `<pad>`、`<unk>` 并重新归一化；这里的概率与原始全词表概率略有区别。


In [ ]:
from src.generation import load_model, predict_next, generate_text
from src.preprocessing import tokenize

loaded_model, loaded_vocab, checkpoint = load_model(result["checkpoint_path"], device)
prefix = "the time traveller"
unknown_input = sorted(set(tokenize(prefix)) - set(loaded_vocab))
print("Prefix:", prefix)
print("输入中的未登录词:", unknown_input)
display(pd.DataFrame(predict_next(loaded_model, loaded_vocab, prefix, top_k=10)))


## 8. 自回归续写

先读取 prefix 得到隐藏状态，然后预测一个新单词，将其作为下一步输入。
生成时保留隐藏状态，LSTM 同时保留 hidden 和 cell。
贪心生成选择最高分单词；温度采样能展示其他候选，但不保证文本流畅。
`top_k=0` 表示不限制普通单词候选数。


In [ ]:
print("贪心生成:")
print(generate_text(loaded_model, loaded_vocab, prefix, num_words=40, temperature=0))
print("\n温度采样:")
print(generate_text(loaded_model, loaded_vocab, prefix, num_words=40, temperature=0.8, top_k=20, seed=config.seed))


## 9. 比较已经训练的模型

另外两个 Notebook 运行后，这里会自动显示三个模型的结果。
比较函数检查语料、词表和超参数是否一致，并按验证困惑度排序。
参数量不同也会列出；训练耗时还受设备影响，不适合跨设备直接比较。
若修改配置，请为三个模型都使用相同配置和独立的输出根目录。


In [ ]:
from src.plotting import comparison_table

comparison = comparison_table(output_dir)
display(comparison)
comparison.to_csv(output_dir / "comparison.csv", index=False)
print("尚未训练的模型不会出现在表中。")


## 代码入口

模型：`src/model_LSTM.py`；训练：`src/engine.py`；生成：`src/generation.py`。
在命令行可以运行 `python train.py --model lstm`，实现与本 Notebook 一致的训练。
PyTorch 接口参考：[nn.LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)。
